# 소방청 화재발생 통계 × 인구 데이터 분석 프로젝트
## 0단계 - 데이터 소스 선정 및 검증

In [1]:
import pandas as pd
import numpy as np
import re

## 1.1 화재 데이터 로드

In [22]:
fire = pd.read_csv(r'C:\data\소방청_화재발생_정보_20241231.csv', encoding='cp949')
print(f'행 : {fire.shape[0]}, 열 : {fire.shape[1]}')

행 : 191510, 열 : 15


## 1.2 구조 파악

In [20]:
print(fire.info())
print('시도(', fire['시도'].nunique(), '개) :', sorted(fire['시도'].unique()))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 191509 entries, 0 to 191508
Data columns (total 15 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   화재발생년원일    191509 non-null  datetime64[ns]
 1   시도         191509 non-null  object        
 2   시군구        191509 non-null  object        
 3   화재유형       191509 non-null  object        
 4   발화요인대분류    191509 non-null  object        
 5   발화요인소분류    191509 non-null  object        
 6   인명피해(명)소계  191509 non-null  int64         
 7   사망         191509 non-null  int64         
 8   부상         191509 non-null  int64         
 9   재산피해소계     191509 non-null  int64         
 10  부동산        191509 non-null  float64       
 11  동산         191509 non-null  float64       
 12  장소대분류      191509 non-null  object        
 13  장소중분류      191509 non-null  object        
 14  장소소분류      191509 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(4), object(8)
memory usage: 

## ── 1.3 화재 데이터 정제 ──

In [19]:
fire['화재발생년원일'] = pd.to_datetime(fire['화재발생년원일'])
print('날짜 범위 :', fire['화재발생년원일'].min(), '~', fire['화재발생년원일'].max())

fire[['부동산', '동산']] = fire[['부동산', '동산']].fillna(0) # 재산피해 0원인 정상 결측
fire = fire.drop_duplicates().reset_index(drop=True) # 완전중복행 제거
fire['시군구'] = fire['시군구'].str.replace(' ', '', regex=False) # 표기 공백 통일 

날짜 범위 : 2020-01-01 00:03:00 ~ 2024-12-31 23:54:00


## 1.4 시군구 단위 집계

In [18]:
fire_agg = (
    fire.groupby(['시도', '시군구'])
    .agg(화재건수=('화재발생년원일', 'count'),
         사망자수=('사망', 'sum'),
         부상자수=('부상', 'sum'),
         재산피해합계_천원=('재산피해소계', 'sum'))
    .reset_index()
)
print('화재 집계:', fire_agg.shape)

화재 집계: (254, 6)


In [6]:
fire_agg

,시도,시군구,화재건수,사망자수,부상자수,재산피해합계_천원
0,강원특별자치도,강릉시,1186,9,96,83383940
1,강원특별자치도,고성군,336,0,16,4122182
2,강원특별자치도,동해시,339,6,12,3988975
3,강원특별자치도,삼척시,356,7,12,8717131
4,강원특별자치도,속초시,400,2,37,10303772
...,...,...,...,...,...,...
249,충청북도,청주시상당구,503,7,90,3486404
250,충청북도,청주시서원구,498,6,43,31767125
251,충청북도,청주시청원구,686,8,57,22430506
252,충청북도,청주시흥덕구,717,2,29,15701291


## 1.4-2 인구 데이터 로드 및 정제

In [16]:
def load_population(filepath):
    df = pd.read_csv(filepath, encoding='cp949')

    # "서울특별시 종로구 (1111000000)" -> "서울특별시 종로구"
    df['지역명'] = df['행정구역'].apply(lambda s: re.sub(r'\s*\(\d+\)\s*$', '', s).strip())

    # 출장소는 실제 시군구 단위가 아니므로 제외
    df = df[~df['지역명'].str.contains('출장소')].copy()

    # 시도 / 시군구 분리 (하위 구는 공백 없이 붙여서 화재 데이터와 표기 통일)
    tok = df['지역명'].str.split()
    df['시도'] = tok.apply(lambda x: x[0])
    df['시군구'] = tok.apply(lambda x: ''.join(x[1:]) if len(x) > 1 else '__TOTAL__')

    # 2023.6 강원도->강원특별자치도, 2024.1 전라북도 ->전북특별자치도 개편
    # (군위군의 경북->대구 편입은 실제 관할 이동이라 여기서 던드리지 않습니다)
    df['시도'] = df['시도'].replace({'강원도': '강원특별자치도', '전라북도': '전북특별자치도'})

    # 세종 특례 : 시도 총계행을 시군구 '세종'으로도 사용
    sejong = (df['시도'] == '세종특별자치시') & (df['시군구'] == '__TOTAL__')
    df.loc[sejong, '시군구'] = '세종'
    df = df[df['시군구'] != '__TOTAL__']

    pop_cols = [c for c in df.columns if '총인구수' in c]
    for c in pop_cols:
        df[c] = pd.to_numeric(df[c].astype(str).str.replace(',', ''), errors='coerce').fillna(0)

    # 개편으로 나뉜 행을 시도+시군구 기준 합산 (해당 없는 기간은 0이라 sum하면 연속값 복원)
    return df.groupby(['시도', '시군구'])[pop_cols].sum().reset_index(), pop_cols

pop1, cols1 = load_population(r'C:\data\202001_202212_주민등록인구및세대현황_월간.csv')
pop2, cols2 = load_population(r'C:\data\202301_202412_주민등록인구및세대현황_월간.csv')

pop_all = pop1.merge(pop2, on =['시도', '시군구'], how='outer')
pop_all['평균인구_5년'] = pop_all[cols1 + cols2].mean(axis=1).round(2)
pop_final = pop_all[['시도', '시군구', '평균인구_5년']]
print('인구 집계 :', pop_final.shape)

인구 집계 : (265, 3)


In [10]:
pop_final

,시도,시군구,평균인구_5년
0,강원특별자치도,강릉시,211402.15
1,강원특별자치도,고성군,27112.33
2,강원특별자치도,동해시,89429.30
3,강원특별자치도,삼척시,63830.78
4,강원특별자치도,속초시,82248.22
...,...,...,...
260,충청북도,청주시상당구,194210.83
261,충청북도,청주시서원구,191282.87
262,충청북도,청주시청원구,192875.42
263,충청북도,청주시흥덕구,270213.40


## 1.5 최종 병합 및 파생변수

In [15]:
analysis = fire_agg.merge(pop_final, on=['시도', '시군구'], how='left')
print('병합 안 된 지역 수 :', analysis['평균인구_5년'].isna().sum()) # 0이어야 정상

analysis['화재발생율_10만명당'] = (analysis['화재건수'] / analysis['평균인구_5년'] * 100000).round(2)
print(analysis.sort_values('화재발생율_10만명당', ascending=False).head())

# SAS로 넘길 파일 저장 (칼럼명은 영문/ASCII로 - 32바이트 문제 재발 방지)
analysis.rename(columns={
    '시도': 'SIDO', '시군구': 'SIGUNGU', '화재건수': 'FIRE_CNT',
    '사망자수': 'DEATH_CNT', '부상자수': 'INJURY_CNT',
    '재산피해합계_천원': 'DMG_TOTAL_SUM', '평균인구_5년': 'POP_AVG_5YR',
    '화재발생율_10만명당': 'FIRE_RATE_100K'
}).to_csv(r'C:\data\fire_population_analysis.csv', index=False, encoding='utf-8-sig')

병합 안 된 지역 수 : 0
       시도  시군구  화재건수  사망자수  부상자수  재산피해합계_천원   평균인구_5년  화재발생율_10만명당
72   경상남도  의령군   428     3    13    4207679  26143.18      1637.14
204  전라남도  함평군   502     4    10    5509405  31180.20      1610.00
87   경상북도  고령군   455     1    18    4362383  30683.92      1482.86
70   경상남도  산청군   504     4    21    6312527  34264.27      1470.92
199  전라남도  영암군   774     5    15    8793031  52905.27      1462.99


## 1.6 SAS 연동용 파일 내보내기
SAS에서 직접 집계·조인 연습을 할 수 있도록 (1) 정제만 된 화재 원자료와
(2) 인구 요약 테이블을 각각 영문 컬럼명으로 내보낸다. 

In [14]:
# (1) 화재 원자료 - 정제만 하고 집계는 SAS에서 직접
rename_map = {
    '화재발생년원일': 'FIRE_DT', '시도': 'SIDO', '시군구': 'SIGUNGU',
    '화재유형': 'FIRE_TYPE', '발화요인대분류': 'CAUSE_L', '발화요인소분류': 'CAUSE_S',
    '인명피해(명)소계': 'CASUALTY', '사망': 'DEATH', '부상': 'INJURY',
    '재산피해소계': 'DMG_TOTAL', '부동산': 'DMG_REAL', '동산': 'DMG_MOVABLE',
    '장소대분류': 'PLACE_L', '장소중분류': 'PLACE_M', '장소소분류': 'PLACE_S'
}
fire.rename(columns=rename_map).to_csv(
    r'C:\data\fire_clean_for_sas.csv', index=False, encoding='utf-8-sig'
)

# (2) 인구 요약 테이블
pop_final.rename(columns={
    '시도': 'SIDO', '시군구': 'SIGUNGU', '평균인구_5년': 'POP_AVG_5YR'
}).to_csv(r'C:\data\population_sigungu_avg.csv', index=False, encoding='utf-8-sig')